# Ecommerce OPK PydanticAI Migration Process

## Initialization

In [189]:
import os
import concurrent.futures
from datetime import datetime
from httpx import AsyncClient
from dotenv import load_dotenv
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Awaitable, Type
from abc import ABC, abstractmethod

load_dotenv()

True

Defining the class models

In [190]:
class ProductCategory(BaseModel):
	id: int
	name: str
	description: str
	product_count: int
	created_at: datetime
	updated_at: Optional[datetime]

class Product(BaseModel):
	id: int
	name: str
	description: str
	product_variant_count: int
	product_category_id: int
	created_at: datetime
	updated_at: Optional[datetime]

class ProductVariant(BaseModel):
	id: int
	name: str
	price: float
	stock: int
	product_id: int
	created_at: datetime
	updated_at: Optional[datetime]

class User(BaseModel):
	id: int
	name: str
	email: str
	phone_number: str
	address_count: int
	cart_item_count: int
	created_at: datetime
	updated_at: Optional[datetime]

class Address(BaseModel):
	id: int
	user_id: int
	address_line: str
	city: str
	state: str
	postal_code: str
	country: str
	created_at: datetime
	updated_at: Optional[datetime]

class CartItem(BaseModel):
	id: int
	quantity: int
	user_id: int
	product_variant_id: int
	created_at: datetime
	updated_at: Optional[datetime]

## Execution

### API HTTP Client

Creating an API HTTP client interface

In [208]:
class CoreApiHttpClientABC(ABC):
	@abstractmethod
	async def fetch_product_categories(self) -> List[ProductCategory]:
		pass

	@abstractmethod
	async def fetch_products_by_product_category(self, product_category_id: int) -> List[Product]:
		pass

	@abstractmethod
	async def fetch_product(self, product_id: int) -> Product:
		pass

	@abstractmethod
	async def fetch_product_variants_by_product(self, product_id: int) -> List[ProductVariant]:
		pass

	@abstractmethod
	async def fetch_user_by_email(self, email: str) -> Optional[User]:
		pass

	@abstractmethod
	async def fetch_user_addresses(self, user_id: int) -> List[Address]:
		pass

	@abstractmethod
	async def fetch_user_cart_items(self, user_id: int) -> List[CartItem]: 
		pass

API HTTP client concrete implementatoin

In [221]:
class CoreApiHttpClient(CoreApiHttpClientABC):
	def __init__(self):
		base_url: str = os.getenv("NOCODB_API_BASE_URL")
		access_token: str = os.getenv("NOCODB_XC_TOKEN")
		headers: Dict[str, str] = {"accept": "application/json", "xc-token": access_token}

		self._http_client = AsyncClient(base_url=base_url, headers=headers)

	async def fetch_product_categories(self) -> Awaitable[List[ProductCategory]]:
		end_point = "/api/v2/tables/m8myg8hob1pyii0/records"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of product categories
		unparsed_product_categories = response.json().get('list')

		# Parse each ProductCategory object into a ProductCategory object
		def parse_product_category(unparsed_product_category: Dict[str, Any]) -> ProductCategory:
			return ProductCategory(
				id=unparsed_product_category.get('Id'),
				name=unparsed_product_category.get('Name'),
				description=unparsed_product_category.get('Description'),
				product_count=unparsed_product_category.get('Products'),
				created_at=unparsed_product_category.get('CreatedAt'), 
				updated_at=unparsed_product_category.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the ProductCategory objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			parsed_product_categories = list(executor.map(parse_product_category, unparsed_product_categories))

		return parsed_product_categories

	async def fetch_products_by_product_category(self, product_category_id: int) -> Awaitable[List[Product]]:
		include_fields = ["Id", "Title", "Name", "Description", "ProductCategories_id", "ProductVariants", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/m8myg8hob1pyii0/links/cbtac8lok4eeekr/records/{product_category_id}?fields={','.join(include_fields)}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()
		
		# Get the list of products
		unparsed_products = response.json().get('list')

		# Parse each Product object into a Product object
		def parse_product(unparsed_product: Dict[str, Any]) -> Product:
			return Product(
				id=unparsed_product.get('Id'),
				name=unparsed_product.get('Name'),
				description=unparsed_product.get('Description'),
				product_variant_count=unparsed_product.get('ProductVariants'),
				product_category_id=unparsed_product.get('ProductCategories_id'),
				created_at=unparsed_product.get('CreatedAt'), 
				updated_at=unparsed_product.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Product objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			parsed_products = list(executor.map(parse_product, unparsed_products))

		return parsed_products

	async def fetch_product(self, product_id: int) -> Awaitable[Product]:
		end_point = f"/api/v2/tables/m9z4v3gwund7k0y/records/{product_id}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Parse the Product object into a Product object
		def parse_product(unparsed_product: Dict[str, Any]) -> Product:
			return Product(
				id=unparsed_product.get('Id'),
				name=unparsed_product.get('Name'),
				description=unparsed_product.get('Description'),
				product_variant_count=unparsed_product.get('ProductVariants'),
				product_category_id=unparsed_product.get('ProductCategories_id'),
				created_at=unparsed_product.get('CreatedAt'), 
				updated_at=unparsed_product.get('UpdatedAt'),	
			)

		return parse_product(response.json())
		pass

	async def fetch_product_variants_by_product(self, product_id: int) -> Awaitable[List[ProductVariant]]:
		include_fields = ["Id", "Title", "Name", "Description", "Price", "Stock", "Products_id", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/m9z4v3gwund7k0y/links/cnwqa1j9ql3ihd0/records/{product_id}?fields={','.join(include_fields)}"
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of product variants
		unparsed_product_variants = response.json().get('list')

		# Parse each ProductVariant object into a ProductVariant object
		def parse_product_variant(unparsed_product_variant: Dict[str, Any]) -> ProductVariant:
			return ProductVariant(
				id=unparsed_product_variant.get('Id'),
				name=unparsed_product_variant.get('Name'),
				price=unparsed_product_variant.get('Price'),
				stock=unparsed_product_variant.get('Stock'),
				product_id=unparsed_product_variant.get('Products_id'),
				created_at=unparsed_product_variant.get('CreatedAt'), 
				updated_at=unparsed_product_variant.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the ProductVariant objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			product_variants = list(executor.map(parse_product_variant, unparsed_product_variants))

		return product_variants

	async def fetch_user_by_email(self, email: str) -> Optional[User]:
		end_point = f"/api/v2/tables/mtkw0ob3dxznnoy/records?where=(Email,eq,{email})&limit=1" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of user
		unparsed_users = response.json().get('list')

		# Parse user into a User object
		def parse_user(unparsed_user: Dict[str, Any]) -> User:
			return User(
				id=unparsed_user.get('Id'),
				name=unparsed_user.get('Name'),
				email=unparsed_user.get('Email'),
				phone_number=unparsed_user.get('PhoneNumber'),
				address_count=unparsed_user.get('Addresses'),
				cart_item_count=unparsed_user.get('CartItems'),
				created_at=unparsed_user.get('CreatedAt'), 
				updated_at=unparsed_user.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the User objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			users = list(executor.map(parse_user, unparsed_users))

		user = users[0] if users else None
		return user

	async def fetch_user_addresses(self, user_id: int) -> List[Address]:
		include_fields = ["Id", "Users_id", "AddressLine", "City", "State", "PostalCode", "Country", "CreatedAt", "UpdatedAt"]
		end_point = f"/api/v2/tables/mtkw0ob3dxznnoy/links/caph3486tt4sx7o/records/{user_id}?fields={",".join(include_fields)}" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of address
		unparsed_addresses = response.json().get('list')

		# Parse address into a Address object
		def parse_address(unparsed_address: Dict[str, Any]) -> Address:
			return Address(
				id=unparsed_address.get('Id'),
				user_id=unparsed_address.get('Users_id'),
				address_line=unparsed_address.get('AddressLine'),
				city=unparsed_address.get('City'),
				state=unparsed_address.get('State'),
				country=unparsed_address.get('Country'),
				postal_code=unparsed_address.get('PostalCode'),
				created_at=unparsed_address.get('CreatedAt'), 
				updated_at=unparsed_address.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Address objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			addresses = list(executor.map(parse_address, unparsed_addresses))

		return addresses
	
	async def fetch_user_cart_items(self, user_id: int) -> List[CartItem]: 
		include_fields = ["Id", "Users_id", "ProductVariants_id", "Quantity", "CreatedAt", "UpdatedAt"] 
		end_point = f"/api/v2/tables/mtkw0ob3dxznnoy/links/crsqehtu2hhjmr4/records/{user_id}?fields={",".join(include_fields)}" 
		response = await self._http_client.get(end_point)
		response.raise_for_status()

		# Get the list of cart_item
		unparsed_cart_items = response.json().get('list')

		# Parse cart_item into a Cart_item object
		def parse_cart_item(unparsed_cart_item: Dict[str, Any]) -> CartItem:
			return CartItem(
				id=unparsed_cart_item.get('Id'),
				user_id=unparsed_cart_item.get('Users_id'),
				product_variant_id=unparsed_cart_item.get('ProductVariants_id'),
				quantity=unparsed_cart_item.get('Quantity'),
				created_at=unparsed_cart_item.get('CreatedAt'), 
				updated_at=unparsed_cart_item.get('UpdatedAt'),	
			)

		# Use a ThreadPoolExecutor to parallelize the parsing of the Cart_item objects
		with concurrent.futures.ThreadPoolExecutor() as executor:
			cart_items = list(executor.map(parse_cart_item, unparsed_cart_items))

		return cart_items
		

#### Testing the functions

In [222]:
core_api_http_client: CoreApiHttpClientABC = CoreApiHttpClient()

In [223]:
await core_api_http_client.fetch_product_categories()

[ProductCategory(id=1, name='Drum Handling Equipment', description='Drum handling equipment are used to safely and efficiently move, lift and pour to discharge drum content. They are the solutions to address every drum handling issues as unsafe handling can result in cost of content or any unnecessary injuries or damages. \n\nDrum handling equipment are particularly used in factories which involve drums in their daily operation food & beverage, & gas, pharmaceutical; chemical; printing industries & etc. They are also used in logistic centres which are involved in distributing drums.', product_count=4, created_at=datetime.datetime(2025, 1, 30, 16, 4, 33, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 16, 10, 1, tzinfo=TzInfo(UTC))),
 ProductCategory(id=2, name='Stacker', description='Stackers are economical alternative to a forklift, commonly used to transfer load onto low or mid-level racking; making the tasks quicker and safer. Because they are smaller in size, stacker

In [224]:
await core_api_http_client.fetch_products_by_product_category(product_category_id=1)

[Product(id=3, name='Drum Handler', description='Strong gripping mechanism.\nDrum is safely supported while tilting.\nLight and smooth tilting even when drum is full loaded.\nCompact and maneuverable.\nTough; reliable and productive.', product_variant_count=1, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 58, 26, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 15, 2, 57, tzinfo=TzInfo(UTC))),
 Product(id=4, name='Hydraulic Drum Porter', description='Strong gripping mechanism.  \nLightweight and effortless maneuvering.\nEasy to operate.\nLow maintenance cost.\nReliable; safe and efficient.', product_variant_count=4, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 15, 3, 3, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 15, 7, 30, tzinfo=TzInfo(UTC))),
 Product(id=1, name='Drum Gripper', description="Auto handling - entire operation s controlled from the driver's seat.\nStrong gripping mechanism, drum is handled safel

In [225]:
await core_api_http_client.fetch_product(product_id=1)

Product(id=1, name='Drum Gripper', description="Auto handling - entire operation s controlled from the driver's seat.\nStrong gripping mechanism, drum is handled safely and efficiently (for N series)\nCradle belt protects drum from dents or scratches (for N Series)\nDrum is grasped securely by the rim of the drum (for U Series)\nJaw can be adjusted manually to fit the diameter of the drum (for U Series)\nEasy fast and safe", product_variant_count=4, product_category_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 13, 1, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 14, 58, 43, tzinfo=TzInfo(UTC)))

In [226]:
await core_api_http_client.fetch_product_variants_by_product(product_id=1)

[ProductVariant(id=1, name='Auto N-1', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 39, 1, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 14, 14, 30, tzinfo=TzInfo(UTC))),
 ProductVariant(id=3, name='Auto U-1', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 41, 3, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 22, 10, tzinfo=TzInfo(UTC))),
 ProductVariant(id=4, name='Auto U-2', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 41, 28, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 1, 30, 14, 59, 12, tzinfo=TzInfo(UTC))),
 ProductVariant(id=2, name='Auto N-2', price=500.0, stock=10, product_id=1, created_at=datetime.datetime(2025, 1, 30, 14, 40, 12, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 24, 22, tzinfo=TzInfo(UTC)))]

In [227]:
await core_api_http_client.fetch_user_by_email(email="david@gmail.com")

User(id=41, name='david delacroz', email='david@gmail.com', phone_number='+639292557199', address_count=1, cart_item_count=0, created_at=datetime.datetime(2025, 2, 4, 12, 44, 19, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 24, 28, tzinfo=TzInfo(UTC)))

In [228]:
await core_api_http_client.fetch_user_addresses(user_id=41)

[Address(id=27, user_id=41, address_line='45 Maple Lane, Queens, New York', city='Queens', state='New York', postal_code='4002', country='USA', created_at=datetime.datetime(2025, 2, 4, 16, 12, 10, tzinfo=TzInfo(UTC)), updated_at=datetime.datetime(2025, 2, 4, 16, 14, 45, tzinfo=TzInfo(UTC)))]

In [230]:
await core_api_http_client.fetch_user_cart_items(user_id=41)

[]